# Part 6: SciML parameter estimation with PEtab.jl

Hybrid scientific machine learning (SciML) combines mechanistic models, such as ODEs, with machine learning models. PEtab.jl in conjunction with Catalyst supports three SciML problem types, as well as combinations of them:

1. An ML model embedded in the ODE dynamics. This includes both Universal Differential Equations (UDEs) and Neural ODEs.
2. An ML model in the observable formula that maps model output to measurement data.
3. A pre-simulation ML model that maps input data, such as high-dimensional images, to ODE parameters and/or initial conditions before simulation.

This notebook focuses on the first case: defining and training a UDE. As a running example, we use simulated data from a Lotka-Volterra model:

$$
\frac{\mathrm{d} \mathrm{prey}}{\mathrm{d} t} = \alpha \cdot \mathrm{prey} - \beta \cdot \mathrm{prey} \cdot \mathrm{predator} \\
\frac{\mathrm{d} \mathrm{predator}}{\mathrm{d} t} = \gamma \cdot \mathrm{prey} \cdot \mathrm{predator} - \delta \cdot \mathrm{predator} \\
$$

Where we replace and learn the interaction term with a neural network:

$$
\frac{\mathrm{d} \mathrm{prey}}{\mathrm{d} t} = \alpha \cdot \mathrm{prey} - NN[1] \\
\frac{\mathrm{d} \mathrm{predator}}{\mathrm{d} t} = NN[2] - \delta \cdot \mathrm{predator} \\
$$

## Part 6.1: Defining a SciML PEtab problem

A SciML `PEtabODEProblem` is constructed largely in the same way as for a purely mechanistic model. The main difference is that ML components are introduced by (1) defining one or more Lux.jl neural networks and (2) wrapping them as `MLModel`s to specify how they interact with the ODE model.

### Part 6.1.1: Defining the ML model

PEtab.jl currently supports ML models defined with [Lux.jl](https://lux.csail.mit.edu/stable/). To be compatible with PEtab.jl, the Lux model must define a set of layers with unique names together with a forward pass. This is most easily done with `Lux.Chain`:

In [ ]:
using Lux
lux_model1 = Lux.Chain(
    Lux.Dense(2 => 5, Lux.swish),
    Lux.Dense(5 => 5, Lux.swish),
    Lux.Dense(5 => 2)
)

Next, the ML-model must be wrapped as an `MLModel` and given a unique ID

In [ ]:
using PEtab
ml_model = MLModel(:net1, lux_model1, false)

The third argument specifies whether the ML model is evaluated before simulation. Here, the ML model is evaluated during simulation, as it enters the ODE dynamics, so this argument is set to `false`.

### Part 6.1.2: Defining the dynamic UDE model

A UDE is constructed by embedding one or more `MLModel`s in the ODE dynamics. At present, UDE models must be provided as an `ODEProblem`. Support for Catalyst-based UDE models is under development, so stay tuned!

The first step is to define the ODE right-hand side. Compared to a standard in-place ODE function, the right-hand side also takes an `ml_models` argument, which stores the declared ML models and can be indexed by their IDs:

In [ ]:
function lv_ude!(du, u, p, t, ml_models)
    prey, predator = u
    net1 = ml_models[:net1]
    nn_out, _ = net1.lux_model([prey, predator], p.net1, net1.st)
    du[1] = p.alpha * prey - nn_out[1] # prey
    du[2] = nn_out[2] - p.delta * predator # predator
    return nothing
end

using ComponentArrays
p_mechanistic = ComponentArray(alpha = 1.3, delta = 1.8)
u0 = ComponentArray(prey = 0.44249296, predator = 4.6280594)
ude_problem = UDEProblem(
    lv_ude!, u0, (0.0, 10.0), p_mechanistic, ml_model
)

The parameter vector `p` and initial conditions `u0` are expected to be `ComponentArray`s so that PEtab.jl can track parameter and state names. This also allows entries to be accessed by name, for example `p.delta` above.

### Part 6.1.3: Parameters to estimate

A `PEtabMLParameter` must be defined for each `MLModel` whose parameters should be estimated. Therefore, when specifying the parameters to estimate, both mechanistic parameters (`PEtabParameter`) and ML parameters (`PEtabMLParameter`) must be declared:

In [ ]:
pest = [
    PEtabMLParameter(:net1),
    PEtabParameter(:alpha; estimate = false, value = 1.3),
    PEtabParameter(:delta; estimate = false, value = 1.8)
]

For simplicity, in this case we keep the mechanistic parameters fixed.

### Part 6.1.4: Measurements and observables

Measurements and observables are defined in the same way as for a mechanistic `PEtabODEProblem`. For this example, we simulate data and split it into training and validation sets:

In [ ]:
using DataFrames, StableRNGs, Statistics, OrdinaryDiffEqTsit5
rng = StableRNGs.StableRNG(42)

function lv_ode!(du, u, p, t)
    prey, predator = u
    du[1] = p.alpha * prey - p.beta * prey * predator
    du[2] = p.gamma * prey * predator - p.delta * predator
    return nothing
end

u0 = [0.44249296, 4.6280594]
p_true = (alpha = 1.3, beta = 0.9, gamma = 0.8, delta = 1.8)
lv_prob = ODEProblem(lv_ode!, u0, (0.0, 13.0), p_true)
sol = solve(lv_prob, Tsit5(); abstol = 1e-8, reltol = 1e-8, saveat = 0:0.1:12.0)

obs_prey = sol[1, :] .+  0.1 .* randn(rng, length(sol.t))
obs_predator = sol[2, :] .+ 0.1 .* randn(rng, length(sol.t))

df_prey = DataFrame(
    time = sol.t, measurement = obs_prey, obs_id = "obs_prey"
)
df_predator = DataFrame(
    time = sol.t, measurement = obs_predator, obs_id = "obs_predator"
)
df_m = vcat(df_prey, df_predator)

t_split = 6.0
df_train = filter(row -> row.time <= t_split, df_m)
df_val = filter(row -> row.time > t_split, df_m)


Plotting the training/validation split shows that this is a challenging training task due to the oscillatory dynamics:

In [ ]:
using Plots
scatter(df_m.time, df_m.measurement, group = df_m.obs_id)
vline!(
    [t_split], label = "split train/validation", color = "black"
)

Since both `prey` and `predator` are assumed measured, the observables are:

In [ ]:
observables = [
    PEtabObservable(:obs_prey, :prey, 1.0),
    PEtabObservable(:obs_predator, :predator, 1.0),
]

### Part 6.1.5: Build the parameter estimation problem

Given training and validation data, separate `PEtabODEProblem`s can be created for the training and validation objectives:

In [ ]:
model_train = PEtabModel(
    ude_problem, observables, df_train, pest; ml_models = ml_model
)
model_val = PEtabModel(
    ude_problem, observables, df_val, pest; ml_models = ml_model
)
prob_train = PEtabODEProblem(model_train)
prob_val = PEtabODEProblem(model_val)

## Part 6.2: Model training (parameter estimation)

SciML problems can be trained with the same methods used for purely mechanistic models, such as multistart local optimization with a BFGS-based optimizer. In practice, training often works better with optimizers commonly used in machine learning, such as Adam. Still, Adam alone is often not sufficient for reliable training.

To address this, several specialized training strategies have been developed. Via [PEtabTraining.jl](https://github.com/sebapersson/PEtabTraining.jl), PEtab.jl supports several efficient approaches: such as curriculum learning. In this part, we focus on training with Adam and curriculum learning.

### Part 6.2.1: Generate start guesses

Regardless of training strategy, optimization requires an initial guess. For SciML problems, such as Neural ODEs and UDEs, training is often more stable when the ML model is initialized with small weights and biases. Otherwise, the initial dynamics can be difficult to simulate. To support this, custom weight and bias initializers can be passed to `get_startguesses`:

In [ ]:
rng = StableRNGs.StableRNG(23) # for reproducibility
# default is glorot_normal(; gain = 1.0)
x0 = get_startguesses(
    rng, prob_train, 1; init_bias = Lux.zeros64,
    init_weight = glorot_normal(; gain = 0.1),
)

### Part 6.2.2: Plain Adam training

Given an initial guess, the objective and its gradient can be evaluated with `prob_train.nllh(x)` and `prob_train.grad(x)`. These can then be used to implement a training loop:

In [ ]:
using Optimisers

x = deepcopy(x0)
learning_rate = 1.5e-3
state = Optimisers.setup(Adam(learning_rate), x)

trace_adam = Float64[]
n_epochs = 14000
for epoch in 1:n_epochs
    g = prob_train.grad(x)
    state, x = Optimisers.update(state, x, g)

    # Stop if the objective cannot be evaluated (e.g. simulation failure)
    nllh_train = prob_train.nllh(x)
    if !isfinite(nllh_train) || nllh_train > 1e10
        break
    end

    # Save training trace for plotting later
    if epoch % 25 == 0 || epoch == 1
        push!(trace_adam, nllh_train)
    end
end
# Final likelihood value
@show prob_train.nllh(x)

Plotting the fit shows an okay fit, but there is still a bit of room for improvement, especially in terms of validation:

In [ ]:
p1 = plot(x, prob_train)
title!(p1, "Training data")
p2 = plot(x, prob_val)
title!(p2, "Validation data")
plot(p1, p2, layout = (1, 2), size = (1400, 400))

### Part 6.2.2: Curriculum learning

Curriculum learning gradually increases the difficulty of the training problem across a sequence of curriculum stages. For a `PEtabODEProblem`, this can be done by first training on a subset of the measurement time points and then progressively including more points until the full dataset is used.

With [PEtabTraining.jl](https://github.com/sebapersson/PEtabTraining.jl), for example, a 5-stage curriculum split over time can be created as:

In [ ]:
using PEtabTraining
prob_cl = PEtabClProblem(prob_train, SplitTime(5))
describe(prob_cl)

`describe(prob_cl)` reports per-stage statistics, such as the fraction of observables and simulation conditions included at each stage. As a rule of thumb, curriculum learning tends to work best when each stage covers most observables and conditions, so that the training objective changes gradually across stages.

The stage-specific problems are stored in `prob_cl.petab_problems` as separate `PEtabODEProblem`s, which can be used to implement a training loop. For example:

In [ ]:
using Optimisers

# Epoch ranges per stage
epochs_per_stage = [
    1 => 1:1000, 2 => 1001:2000, 3 => 2001:3000, 4 => 3001:4000,
    5 => 4001:14000
]

x = deepcopy(x0)
learning_rate = 1.5e-3
state = Optimisers.setup(Adam(learning_rate), x)

trace_cl = Float64[]
for (stage, epochs) in epochs_per_stage
    prob_stage = prob_cl.petab_problems[stage]
    for epoch in epochs
        g = prob_stage.grad(x)
        state, x = Optimisers.update(state, x, g)

        # Stop if the objective cannot be evaluated (e.g. simulation failure)
        nllh_train = prob_train.nllh(x)
        if !isfinite(nllh_train) || nllh_train > 1e10
            break
        end

        # Save training trace (on full training objective for comparability)
        if epoch % 25 == 0 || epoch == 1
            push!(trace_cl, nllh_train)
        end
    end
end
# Final likelihood value
@show prob_train.nllh(x)

Plotting the fit shows an improvement compared to plain Adam:

In [ ]:
p1 = plot(x, prob_train)
title!(p1, "Training data")
p2 = plot(x, prob_val)
title!(p2, "Validation data")
plot(p1, p2, layout = (1, 2), size = (1400, 400))

### Part 6.2.3: Do it yourself: Curriculum multiple shooting

[PEtabTraining.jl](https://github.com/sebapersson/PEtabTraining.jl) also supports multiple shooting (MS) and curriculum multiple shooting (CL+MS). Apply CL+MS to this example and compare the training performance to the results above. A good window penalty value for this model is `1.0`.

## Part 6.2: Bonus: The PEtab-SciML standard format

PEtab.jl SciML functionality is built around the [PEtab-SciML standard](https://github.com/PEtab-dev/petab_sciml) for parameter estimating SciML problems. Models in this format can be imported directly into PEtab.jl. For example, the LV-model we worked with above can be imported as:

In [ ]:
using PEtab
path_yaml = joinpath(@__DIR__, "assets", "lv_ude", "problem.yaml")
ml_models = MLModels(path_yaml)

## Where to go next

This notebook showed how to define a SciML `PEtabODEProblem`. For additional features, including SciML problem types beyond UDEs, see the following tutorials:

- [ML models in observables](https://sebapersson.github.io/PEtab.jl/stable/tutorials/sciml/observable): Define an ML model in a `PEtabObservable` formula, for example to correct model misspecification.
- [Pre-simulation ML models](https://sebapersson.github.io/PEtab.jl/stable/tutorials/sciml/pre_simulate): Define ML models that map input data, such as high-dimensional images, to ODE parameters or initial conditions before simulation.
- [Importing PEtab SciML](https://sebapersson.github.io/PEtab.jl/stable/tutorials/sciml/standard_format): Import problems in the PEtab-SciML standard format.